# **BizFlow360 Model Comparison**
* **By:** Edusei Mikel
* **Date:** 7th August, 2026

**Imports, Data Loading, Preprocessing and Splitting**

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load the synthetic data
df = pd.read_csv('../data/synthetic_msme_data.csv')

# Encode categorical variables (must match training)
le_county = LabelEncoder()
le_sector = LabelEncoder()
df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

# Define Features (X) and Target (y)
features = [
    'business_age_months', 'employees', 
    'monthly_revenue_kes', 'monthly_expenses_kes', 
    'total_assets_kes', 'total_liabilities_kes', 
    'loan_amount_kes', 'mpesa_volume_kes', 
    'county_encoded', 'sector_encoded'
]

X = df[features]
y = df['distress_label']

# Split and Scale (MUST use random_state=42 to match previous notebooks)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Test data loaded and preprocessed successfully.")

✅ Test data loaded and preprocessed successfully.


**Loading all the Trained Models**

In [5]:
# Define the paths to your saved models
model_paths = {
    'Logistic Regression (Baseline)': '../models/trained/on_synthetic_data/logistic_regression_baseline.joblib',
    'Random Forest': '../models/trained/on_synthetic_data/random_forest.joblib',
    'XGBoost': '../models/trained/on_synthetic_data/xgboost.joblib',
    'LightGBM': '../models/trained/on_synthetic_data/lightgbm.joblib'
}

models = {}
for name, path in model_paths.items():
    if os.path.exists(path):
        models[name] = joblib.load(path)
        print(f"✅ Loaded: {name}")
    else:
        print(f"⚠️ Warning: Could not find {path}")

print("\nAll models loaded successfully!")

✅ Loaded: Logistic Regression (Baseline)
✅ Loaded: Random Forest
✅ Loaded: XGBoost
✅ Loaded: LightGBM

All models loaded successfully!


**Evaluation and Building a Comparison Table**

In [6]:
# Function to calculate metrics
def get_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    }

# Evaluate all models
results = {}
for name, model in models.items():
    results[name] = get_metrics(model, X_test_scaled, y_test)

# Create the comparison DataFrame (This is exactly what goes into your Final Report!)
comparison_df = pd.DataFrame(results).T

print("="*70)
print(" FINAL MODEL COMPARISON (Section 3.10)")
print("="*70)
print(comparison_df)
print("="*70)

 FINAL MODEL COMPARISON (Section 3.10)
                                Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression (Baseline)     0.739     0.7294   0.760    0.7444   0.7895
Random Forest                      0.729     0.7421   0.702    0.7215   0.8056
XGBoost                            0.738     0.7380   0.738    0.7380   0.8365
LightGBM                           0.747     0.7426   0.756    0.7493   0.8322


**Saving The Results and Exporting the Best Model**

In [8]:
# Ensure metrics directory exists
os.makedirs('../models/metrics/on_synthetic_data', exist_ok=True)
os.makedirs('../models/trained/on_synthetic_data', exist_ok=True)

# 1. Save the comparison table to CSV
comparison_df.to_csv('../models/metrics/model_comparison.csv')
print("✅ Saved model_comparison.csv to edusei_ml/models/metrics/on_synthetic_data")

# 2. Identify the best model based on ROC-AUC
best_model_name = comparison_df['ROC-AUC'].idxmax()
best_roc_auc = comparison_df['ROC-AUC'].max()
print(f"\n🏆 WINNER: {best_model_name} (ROC-AUC: {best_roc_auc})")

# 3. Copy the winning model to 'best_model.joblib' for Yvette to use in the Streamlit app
import shutil
winning_path = model_paths[best_model_name]
shutil.copy(winning_path, '../models/trained/on_synthetic_data/bizflow_synthetic_v1.0.joblib')
print(f"✅ Copied {best_model_name} to bizflow_synthetic_v1.0.joblib for deployment.")

✅ Saved model_comparison.csv to edusei_ml/models/metrics/on_synthetic_data

🏆 WINNER: XGBoost (ROC-AUC: 0.8365)
✅ Copied XGBoost to bizflow_synthetic_v1.0.joblib for deployment.
